In [0]:
import requests
import json

url = "https://dadosabertos.camara.leg.br/api/v2/eventos"
params = {
    "itens": 5,
    "ordem": "ASC",
    "ordenarPor": "dataHoraInicio"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Total de eventos retornados: {len(dados['dados'])}")
    print(json.dumps(dados, indent=2, ensure_ascii=False)[:3000])
else:
    print(f"Erro: {response.text}")

# Coleta de Eventos Legislativos (2022-2025)
Busca todos os eventos da API da Câmara para o período de 4 anos, incluindo ano eleitoral (2022 e 2024).  
Cada ano é coletado separadamente com paginação automática.

In [0]:
import requests
import json
import time
from datetime import datetime

def coletar_eventos_por_ano(ano, itens_por_pagina=100):
    url = "https://dadosabertos.camara.leg.br/api/v2/eventos"
    todos_eventos = []
    pagina_atual = 1
    
    data_inicio = f"{ano}-01-01"
    data_fim = f"{ano}-12-31"
    
    print(f"Iniciando coleta de eventos para {ano}...")
    
    while True:
        params = {
            "itens": itens_por_pagina,
            "pagina": pagina_atual,
            "dataInicio": data_inicio,
            "dataFim": data_fim,
            "ordem": "ASC",
            "ordenarPor": "dataHoraInicio"
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f"Erro na pagina {pagina_atual} de {ano}: {response.status_code}")
            break
        
        dados = response.json()
        eventos_pagina = dados['dados']
        
        if not eventos_pagina:
            break
        
        todos_eventos.extend(eventos_pagina)
        
        if pagina_atual % 5 == 0 or pagina_atual == 1:
            print(f"  Pagina {pagina_atual}: {len(todos_eventos)} eventos acumulados")
        
        links = dados.get('links', [])
        tem_proxima = any(link['rel'] == 'next' for link in links)
        
        if not tem_proxima:
            break
        
        pagina_atual += 1
        time.sleep(0.2)
    
    print(f"  {ano} concluido! Total: {len(todos_eventos)} eventos\n")
    return todos_eventos

# Coletar 2022, 2023, 2024, 2025
eventos_2022 = coletar_eventos_por_ano(2022)
eventos_2023 = coletar_eventos_por_ano(2023)
eventos_2024 = coletar_eventos_por_ano(2024)
eventos_2025 = coletar_eventos_por_ano(2025)

todos_eventos = eventos_2022 + eventos_2023 + eventos_2024 + eventos_2025
print(f"Total geral de eventos: {len(todos_eventos)}")

# Criação da tabela Bronze de Eventos
Salva os eventos brutos da API na camada bronze com metadados de auditoria.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze

In [0]:
# Preparar dados para a bronze
import json
from datetime import datetime

data_coleta = datetime.now().isoformat()
dados_bronze = []

for evento in todos_eventos:
    linha = (
        evento['id'],
        evento.get('dataHoraInicio'),
        evento.get('dataHoraFim'),
        evento.get('situacao'),
        evento.get('descricaoTipo'),
        evento.get('descricao'),
        evento.get('localExterno'),
        json.dumps(evento.get('orgaos', []), ensure_ascii=False),
        json.dumps(evento.get('localCamara', {}), ensure_ascii=False),
        evento.get('urlRegistro'),
        json.dumps(evento, ensure_ascii=False),
        data_coleta,
        'api_camara_eventos'
    )
    dados_bronze.append(linha)

print(f"Registros preparados: {len(dados_bronze)}")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType

schema = StructType([
    StructField("id_evento", LongType(), True),
    StructField("data_hora_inicio", StringType(), True),
    StructField("data_hora_fim", StringType(), True),
    StructField("situacao", StringType(), True),
    StructField("descricao_tipo", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("local_externo", StringType(), True),
    StructField("orgaos_json", StringType(), True),
    StructField("local_camara_json", StringType(), True),
    StructField("url_registro", StringType(), True),
    StructField("dado_bruto_json", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df = spark.createDataFrame(dados_bronze, schema=schema)

df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.eventos")

print(f"Tabela bronze.eventos criada com {df.count()} registros.")

In [0]:
%sql
SELECT 
    COUNT(*) AS total,
    COUNT(DISTINCT id_evento) AS eventos_unicos,
    MIN(data_hora_inicio) AS primeiro_evento,
    MAX(data_hora_inicio) AS ultimo_evento,
    COUNT(DISTINCT descricao_tipo) AS tipos_evento
FROM bronze.eventos

In [0]:
import requests
import json

# Testar eventos de um deputado especifico
id_deputado = 204379  # Acácio Favacho (primeiro da nossa lista inicial)

url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_deputado}/eventos"
params = {
    "itens": 5,
    "dataInicio": "2024-01-01",
    "dataFim": "2024-12-31",
    "ordem": "ASC",
    "ordenarPor": "dataHoraInicio"
}

response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    dados = response.json()
    print(f"Total de eventos do deputado: {len(dados['dados'])}")
    if dados['dados']:
        print(json.dumps(dados['dados'][0], indent=2, ensure_ascii=False))
    # Ver links de paginacao
    for link in dados.get('links', []):
        print(f"  {link['rel']}: {link['href']}")
else:
    print(f"Erro: {response.text}")

# Coleta de Presença dos Deputados em Eventos (2024)
Busca todos os eventos que cada deputado participou em 2024 usando o endpoint /deputados/{id}/eventos.

In [0]:
import requests
import time
import json
from datetime import datetime
from pyspark.sql.types import StructType, StructField, LongType, StringType

# ============================================================
# ETAPA 1: Coleta de presencas
# ============================================================
df_deputados = spark.sql("""
    SELECT DISTINCT id_deputado, nome_deputado, sigla_partido, sigla_uf
    FROM ouro.frentes_membros_gold
    WHERE sigla_partido IS NOT NULL
""")
deputados = df_deputados.collect()
total_dep = len(deputados)
print(f"Deputados a processar: {total_dep}")

presencas = []
erros = []
inicio = time.time()

for i, dep in enumerate(deputados, start=1):
    id_dep = dep['id_deputado']
    url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_dep}/eventos"
    params = {
        "dataInicio": "2024-01-01",
        "dataFim": "2024-12-31",
        "itens": 200,
        "ordem": "ASC",
        "ordenarPor": "dataHoraInicio"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        dados = response.json()
        eventos_dep = dados.get('dados', [])
        
        for evento in eventos_dep:
            presencas.append((
                id_dep,
                dep['nome_deputado'],
                dep['sigla_partido'],
                dep['sigla_uf'],
                evento['id'],
                evento.get('dataHoraInicio'),
                evento.get('dataHoraFim'),
                evento.get('descricaoTipo'),
                evento.get('situacao'),
                evento.get('descricao')
            ))
    else:
        erros.append(id_dep)
    
    if i % 200 == 0 or i == 1 or i == total_dep:
        decorrido = time.time() - inicio
        print(f"  {i}/{total_dep} deputados | {len(presencas)} presencas | {decorrido:.0f}s")

print(f"\nColeta concluida! {len(presencas)} presencas, {len(erros)} erros")

# ============================================================
# ETAPA 2: Preparar e salvar na Bronze
# ============================================================
data_coleta = datetime.now().isoformat()
dados_presenca = []

for p in presencas:
    linha = (
        p[0], p[1], p[2], p[3], p[4], p[5], p[6], p[7], p[8], p[9],
        data_coleta,
        'api_camara_deputados_eventos'
    )
    dados_presenca.append(linha)

print(f"Registros preparados: {len(dados_presenca)}")

schema = StructType([
    StructField("id_deputado", LongType(), True),
    StructField("nome_deputado", StringType(), True),
    StructField("sigla_partido", StringType(), True),
    StructField("sigla_uf", StringType(), True),
    StructField("id_evento", LongType(), True),
    StructField("data_hora_inicio", StringType(), True),
    StructField("data_hora_fim", StringType(), True),
    StructField("descricao_tipo", StringType(), True),
    StructField("situacao", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

spark.sql("DROP TABLE IF EXISTS bronze.eventos_presenca")

df_presenca = spark.createDataFrame(dados_presenca, schema=schema)
df_presenca.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.eventos_presenca")

print(f"Tabela bronze.eventos_presenca criada com {df_presenca.count()} registros.")

In [0]:
%sql
SELECT 
    COUNT(*) AS total,
    COUNT(DISTINCT id_deputado) AS deputados,
    COUNT(DISTINCT id_evento) AS eventos,
    COUNT(DISTINCT descricao_tipo) AS tipos_evento
FROM bronze.eventos_presenca
